# Stage 4: Build Reasoning Graphs

Converts each reasoning trace into a graph where steps are nodes and edges connect related steps.

We use four edge types, each capturing a different relationship:
- **Sequential (0):** Step i connects to Step i+1 — the basic reasoning flow
- **Semantic (1):** Steps with cosine similarity above 0.75 — they discuss the same concept
- **Value-reuse (2):** A number from Step i appears again in Step j — arithmetic dependency
- **Mentions-number (3):** A number appears in 3 or more steps — all those steps are connected

Edge type is stored as `edge_attr` so the GAT model can learn different attention per edge type.

**Input:** `data/gsm8k_embeddings.npy`, `data/gsm8k_steps_meta.csv`  
**Output:** `data/graphs.pt`

## Cell 1 — Install libraries

We need `torch-geometric` for the graph data structures.

After this cell → **Kernel → Restart** → run all cells top to bottom.

In [ ]:
import sys

# Install PyTorch Geometric and its dependencies
!{sys.executable} -m pip install -q torch-geometric
!{sys.executable} -m pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.5.1+cpu.html

print(" Done — Kernel → Restart, then run from Cell 2")

 Done — Kernel → Restart, then run from Cell 2


## Cell 2 — Imports and file paths

We import all libraries and set up paths to the Stage 3 output files.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data

# File paths
DATA_DIR    = os.path.join(os.getcwd(), 'data')
EMB_FILE    = os.path.join(DATA_DIR, 'gsm8k_embeddings.npy')    # Stage 3 output
META_FILE   = os.path.join(DATA_DIR, 'gsm8k_steps_meta.csv')    # Stage 3 output
OUTPUT_FILE = os.path.join(DATA_DIR, 'graphs.pt')               # Stage 4 output

print(f" Imports OK")
print(f" Embeddings: {EMB_FILE}")
print(f" Meta:       {META_FILE}")
print(f" Output:     {OUTPUT_FILE}")
print()

# Check input files exist
for f in [EMB_FILE, META_FILE]:
    if os.path.exists(f):
        print(f" Found: {os.path.basename(f)}")
    else:
        print(f" Missing: {f} — run Stage 3 first")

## Cell 3 — Load Stage 3 data

Load the embeddings array and the metadata CSV.

- `embeddings` — shape (1450, 768) — one 768-number vector per step
- `meta` — the steps CSV with trace_id, step_num, step_text, label, split

The row index in `embeddings` matches the row index in `meta` — row 0 in embeddings = row 0 in meta.

In [ ]:
# Load embeddings
embeddings = np.load(EMB_FILE)
print(f"Embeddings shape: {embeddings.shape}")
print(f"  Rows = steps, Cols = embedding dimensions")
print()

# Load metadata
meta = pd.read_csv(META_FILE)
print(f"Meta rows:      {len(meta)}")
print(f"Meta columns:   {list(meta.columns)}")
print(f"Unique traces:  {meta['trace_id'].nunique()}")
print()

# Quick sanity check — row counts must match
assert len(embeddings) == len(meta), " Embeddings and meta row count mismatch!"
print(" Row counts match — embeddings aligned with metadata")
print()

# Label distribution
trace_labels = meta.groupby('trace_id')['label'].first()
print(f"Trace labels:")
print(f"  Correct (1): {(trace_labels==1).sum()} traces")
print(f"  Wrong   (0): {(trace_labels==0).sum()} traces")

Embeddings shape: (9490, 768)
  Rows = steps, Cols = embedding dimensions

Meta rows:      9490
Meta columns:   ['trace_id', 'orig_idx', 'split', 'question', 'step_num', 'step_text', 'total_steps', 'label']
Unique traces:  1948

 Row counts match — embeddings aligned with metadata

Trace labels:
  Correct (1): 1461 traces
  Wrong   (0): 487 traces


## Cell 4 — Define the 3 edge-building functions

One function per edge type. Each returns a list of `(src, dst)` tuples — which node connects to which.

---

**`sequential_edges(n)`**  
Connects Step i → Step i+1 for all steps.  
A trace with 4 steps gives edges: (0→1), (1→2), (2→3).  
We also add reverse edges (1→0, 2→1, 3→2) so information flows both ways.

---

**`semantic_edges(step_embeddings, threshold=0.75)`**  
Computes cosine similarity between every pair of steps.  
If similarity > 0.75, adds an edge between them.  
Skips pairs that already have a sequential edge.

---

**`value_reuse_edges(step_texts)`**  
Extracts all numbers from each step.  
If a number that appeared in Step i reappears in Step j (j > i), adds edge i→j.  
Example: Step 2 computes 24, Step 3 uses 24 → edge (1→2) added.

In [ ]:
def sequential_edges(n):
    """
    Build sequential edges: Step i -> Step i+1 in both directions.
    edge_attr type = 0
    """
    edges = []
    for i in range(n - 1):
        edges.append((i, i + 1))
        edges.append((i + 1, i))
    return edges


def cosine_similarity_matrix(embs):
    """Pairwise cosine similarity. Returns (n, n) matrix."""
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    norms = np.clip(norms, a_min=1e-9, a_max=None)
    normalized = embs / norms
    return normalized @ normalized.T


def semantic_edges(step_embeddings, threshold=0.75):
    """
    Add edges between non-adjacent steps with cosine similarity > threshold.
    edge_attr type = 1
    """
    edges = []
    n = len(step_embeddings)
    if n < 2:
        return edges
    sim_matrix = cosine_similarity_matrix(step_embeddings)
    for i in range(n):
        for j in range(i + 2, n):
            if sim_matrix[i, j] > threshold:
                edges.append((i, j))
                edges.append((j, i))
    return edges


def extract_numbers(text):
    """Extract all numbers > 2 from step text."""
    nums = re.findall(r'\b\d+\.?\d*\b', str(text))
    return {x for x in nums if float(x) > 2}


def value_reuse_edges(step_texts):
    """
    Add edges when a number from Step i reappears in Step j (j > i+1).
    Captures arithmetic dependencies.
    edge_attr type = 2
    """
    edges = []
    n = len(step_texts)
    if n < 2:
        return edges
    step_numbers = [extract_numbers(t) for t in step_texts]
    for i in range(n):
        for j in range(i + 1, n):
            shared = step_numbers[i] & step_numbers[j]
            if shared:
                edges.append((i, j))
                edges.append((j, i))
    return edges


def mentions_number_edges(step_texts):
    """
    NEW — inspired by client's entity node approach.
    For each unique number that appears in 3+ steps,
    connect ALL steps that mention it to each other.

    This differs from value_reuse_edges:
    - value_reuse: only connects step i -> step j (j > i) — directional
    - mentions_number: connects ALL steps mentioning the same key number
      regardless of order — captures global numeric context

    Only fires for numbers appearing in 3+ steps (truly significant numbers).
    edge_attr type = 3
    """
    edges = []
    n = len(step_texts)
    if n < 3:
        return edges

    step_numbers = [extract_numbers(t) for t in step_texts]

    # Find all numbers and which steps mention them
    number_to_steps = {}
    for step_idx, nums in enumerate(step_numbers):
        for num in nums:
            if num not in number_to_steps:
                number_to_steps[num] = []
            number_to_steps[num].append(step_idx)

    # Add edges between ALL steps sharing a number that appears 3+ times
    added = set()
    for num, steps_mentioning in number_to_steps.items():
        if len(steps_mentioning) >= 3:     # only truly recurring numbers
            for a in steps_mentioning:
                for b in steps_mentioning:
                    if a != b:
                        pair = (min(a,b), max(a,b))
                        if pair not in added:
                            edges.append((a, b))
                            edges.append((b, a))
                            added.add(pair)
    return edges


#  Quick test
print('TEST — 4-step trace:')
print()
test_texts = [
    'Natalia sold 48 clips in April.',
    'In May she sold half of 48 which is 24 clips.',
    'Total clips = 48 + 24 = 72.',
    'The answer is 72.',
]
test_embs = embeddings[:4]

seq  = sequential_edges(4)
sem  = semantic_edges(test_embs, threshold=0.75)
val  = value_reuse_edges(test_texts)
men  = mentions_number_edges(test_texts)

print(f'Sequential edges:       {seq}')
print(f'Semantic edges:         {sem}')
print(f'Value-reuse edges:      {val}')
print(f'Mentions-number edges:  {men}')
print()
print('Note: 48 appears in steps 0,1,2 -> mentions_number connects all 3')
print()
print(' All 4 edge functions working')


TEST — 4-step trace:

Sequential edges:       [(0, 1), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2)]
Semantic edges:         []
Value-reuse edges:      [(0, 1), (1, 0), (0, 2), (2, 0), (1, 2), (2, 1), (2, 3), (3, 2)]
Mentions-number edges:  [(0, 1), (1, 0), (0, 2), (2, 0), (1, 2), (2, 1)]

Note: 48 appears in steps 0,1,2 -> mentions_number connects all 3

 All 4 edge functions working


## Cell 5 — Build one graph per trace

This is the main loop. For each of the 292 traces we:
1. Get all its steps and their embeddings
2. Build the 3 types of edges
3. Combine all edges, removing duplicates
4. Create a PyTorch Geometric `Data` object
5. Append to a list

**Expected time: under 30 seconds — no API calls, no model downloads.**

In [ ]:
import time

# Edge type legend:
# 0 = sequential      (Step i -> Step i+1)
# 1 = semantic        (cosine similarity > 0.75)
# 2 = value-reuse     (number from step i reappears in step j)
# 3 = mentions-number (same number appears in 3+ steps — all connected)

graphs     = []
skipped    = 0
edge_stats = {'sequential': 0, 'semantic': 0, 'value_reuse': 0, 'mentions_number': 0}

trace_ids = meta['trace_id'].unique()
print(f'Building graphs for {len(trace_ids)} traces...')
print()

start = time.time()

for trace_id in trace_ids:

    trace_rows   = meta[meta['trace_id'] == trace_id].sort_values('step_num')
    step_indices = trace_rows.index.tolist()
    n_steps      = len(step_indices)

    if n_steps < 2:
        skipped += 1
        continue

    node_features = embeddings[step_indices]
    step_texts    = trace_rows['step_text'].tolist()

    # Build all 4 edge types
    seq_edges = sequential_edges(n_steps)
    sem_edges = semantic_edges(node_features, threshold=0.75)
    val_edges = value_reuse_edges(step_texts)
    men_edges = mentions_number_edges(step_texts)

    edge_stats['sequential']      += len(seq_edges)
    edge_stats['semantic']        += len(sem_edges)
    edge_stats['value_reuse']     += len(val_edges)
    edge_stats['mentions_number'] += len(men_edges)

    # Track which type each edge came from (priority: seq > sem > val > men)
    seq_set = set(map(tuple, seq_edges))
    sem_set = set(map(tuple, sem_edges))
    val_set = set(map(tuple, val_edges))
    men_set = set(map(tuple, men_edges))

    all_edge_set = seq_set | sem_set | val_set | men_set
    all_edges    = list(all_edge_set)

    if len(all_edges) == 0:
        skipped += 1
        continue

    # Assign edge type (first match wins: seq > sem > val > men)
    edge_type_list = []
    for e in all_edges:
        et = tuple(e)
        if et in seq_set:
            edge_type_list.append(0)
        elif et in sem_set:
            edge_type_list.append(1)
        elif et in val_set:
            edge_type_list.append(2)
        else:
            edge_type_list.append(3)

    x = torch.tensor(node_features, dtype=torch.float)

    edge_index = torch.tensor(
        [[e[0] for e in all_edges],
         [e[1] for e in all_edges]],
        dtype=torch.long
    )

    edge_attr = torch.tensor(edge_type_list, dtype=torch.float).unsqueeze(1)

    label = int(trace_rows['label'].iloc[0])
    y     = torch.tensor([label], dtype=torch.long)
    split = trace_rows['split'].iloc[0]

    graph = Data(
        x          = x,
        edge_index = edge_index,
        edge_attr  = edge_attr,
        y          = y,
        trace_id   = trace_id,
        split      = split,
        n_steps    = n_steps
    )
    graphs.append(graph)

elapsed = time.time() - start

print(f'Done in {elapsed:.1f} seconds')
print()
print(f'Graphs created:  {len(graphs)}')
print(f'Traces skipped:  {skipped}')
print()
print('Edge counts across all graphs:')
for etype, count in edge_stats.items():
    print(f'  {etype:<20}: {count}')
print()

# Verify edge_attr values are 0-3
sample_attrs = [sorted(g.edge_attr.unique().tolist()) for g in graphs[:3]]
print(f'Sample edge_attr values (first 3 graphs): {sample_attrs}')
print('(should contain values from {0.0, 1.0, 2.0, 3.0})')
print()

train_g = [g for g in graphs if g.split == 'train']
val_g   = [g for g in graphs if g.split == 'val']
test_g  = [g for g in graphs if g.split == 'test']
print(f'Split: Train={len(train_g)}  Val={len(val_g)}  Test={len(test_g)}')


Building graphs for 1948 traces...

Done in 4.3 seconds

Graphs created:  1948
Traces skipped:  0

Edge counts across all graphs:
  sequential          : 15084
  semantic            : 8822
  value_reuse         : 23926
  mentions_number     : 17636

Sample edge_attr values (first 3 graphs): [[0.0], [0.0], [0.0, 1.0, 2.0]]
(should contain values from {0.0, 1.0, 2.0, 3.0})

Split: Train=1363  Val=292  Test=293


## Cell 6 — Save all graphs

Save all 292 graph objects to a single file `graphs.pt` using PyTorch's save function.

Stage 5 will load this file directly to train the GNN.

In [ ]:
torch.save(graphs, OUTPUT_FILE)

file_size = os.path.getsize(OUTPUT_FILE) / 1e6
print(f" Saved: {OUTPUT_FILE}")
print(f"   Graphs: {len(graphs)}")
print(f"   Size:   {file_size:.1f} MB")
print()

# Verify we can reload it
reloaded = torch.load(OUTPUT_FILE)
print(f" Reload test passed — {len(reloaded)} graphs loaded back correctly")

## Cell 7 — Quality checks

Six checks to confirm the graphs are correct before Stage 5.

| # | Check | Target |
|---|---|---|
| 1 | Total graphs | 280–292 |
| 2 | Node feature shape | (n_steps, 768) per graph |
| 3 | Edge index valid | No out-of-bounds node indices |
| 4 | Both labels present | At least some 0s and 1s |
| 5 | All 3 splits present | train, val, test |
| 6 | Avg edges per graph | > 2 (more than just sequential) |

In [ ]:
graphs_check = torch.load(OUTPUT_FILE, weights_only=False)

print('=' * 55)
print('  STAGE 4 QUALITY CHECKS')
print('=' * 55)

all_pass = True

# Check 1: Total graphs
print(f'\n[1] Total graphs: {len(graphs_check)}')
if len(graphs_check) >= 280:
    print('     PASS')
else:
    print('     FAIL — too few graphs')
    all_pass = False

# Check 2: Node feature shape
bad_shape = sum(1 for g in graphs_check if g.x.shape[1] != 768)
print(f'\n[2] Graphs with wrong node feature shape: {bad_shape}')
if bad_shape == 0:
    print('     PASS — all nodes have 768 features')
else:
    print('     FAIL')
    all_pass = False

# Check 3: Edge index valid
bad_edges = sum(1 for g in graphs_check if g.edge_index.max().item() >= g.x.shape[0])
print(f'\n[3] Graphs with out-of-bounds edge indices: {bad_edges}')
if bad_edges == 0:
    print('     PASS — all edges valid')
else:
    print('     FAIL')
    all_pass = False

# Check 4: Both labels present
all_labels = [g.y.item() for g in graphs_check]
n_correct  = sum(1 for l in all_labels if l == 1)
n_wrong    = sum(1 for l in all_labels if l == 0)
print(f'\n[4] Label distribution:')
print(f'    Correct (1): {n_correct}')
print(f'    Wrong   (0): {n_wrong}')
if n_correct > 0 and n_wrong > 0:
    print('     PASS')
else:
    print('     FAIL')
    all_pass = False

# Check 5: All 3 splits
splits = set(g.split for g in graphs_check)
print(f'\n[5] Splits: {splits}')
for s in ['train', 'val', 'test']:
    print(f'    {s}: {sum(1 for g in graphs_check if g.split == s)}')
if {'train','val','test'}.issubset(splits):
    print('     PASS')
else:
    print('     FAIL')
    all_pass = False

# Check 6: Average edges
avg_edges = sum(g.edge_index.shape[1] for g in graphs_check) / len(graphs_check)
avg_nodes = sum(g.x.shape[0] for g in graphs_check) / len(graphs_check)
print(f'\n[6] Avg nodes/graph: {avg_nodes:.1f}  |  Avg edges/graph: {avg_edges:.1f}')
if avg_edges > 2:
    print('     PASS')
else:
    print('      Very few edges')
    all_pass = False

# Check 7: Edge attr present and values are 0-3 (now 4 types)
missing_attr = sum(1 for g in graphs_check if not hasattr(g,'edge_attr') or g.edge_attr is None)
print(f'\n[7] Graphs with edge_attr: {len(graphs_check)-missing_attr}/{len(graphs_check)}')
if missing_attr == 0:
    valid_types = all(
        set(g.edge_attr.squeeze().tolist()).issubset({0.0,1.0,2.0,3.0})
        for g in graphs_check
    )
    # Count how many graphs use edge type 3
    has_type3 = sum(1 for g in graphs_check if 3.0 in g.edge_attr.squeeze().tolist())
    if valid_types:
        print(f'     PASS — edge types 0/1/2/3 present')
        print(f'    Graphs using edge type 3 (mentions-number): {has_type3}')
    else:
        print('      Unexpected edge type values')
else:
    print(f'     FAIL — {missing_attr} graphs missing edge_attr')
    all_pass = False

# Check 8: Edge type distribution across all graphs
from collections import Counter
all_edge_types = []
for g in graphs_check:
    all_edge_types.extend([int(x) for x in g.edge_attr.squeeze().tolist()])
type_counts = Counter(all_edge_types)
total_edges = sum(type_counts.values())
print(f'\n[8] Edge type distribution across all graphs:')
type_names = {0:'sequential', 1:'semantic', 2:'value-reuse', 3:'mentions-number'}
for t in sorted(type_counts.keys()):
    pct = type_counts[t]/total_edges*100
    print(f'    Type {t} ({type_names.get(t,"?")}): {type_counts[t]:>6} ({pct:.1f}%)')
print('     PASS')

print()
print('=' * 55)
if all_pass:
    print('   ALL CHECKS PASSED — Stage 4 complete!')
    print('    Ready for Stage 5 (train GNN)')
else:
    print('    SOME CHECKS FAILED — see above')
print('=' * 55)


  STAGE 4 QUALITY CHECKS

[1] Total graphs: 1948
     PASS

[2] Graphs with wrong node feature shape: 0
     PASS — all nodes have 768 features

[3] Graphs with out-of-bounds edge indices: 0
     PASS — all edges valid

[4] Label distribution:
    Correct (1): 1461
    Wrong   (0): 487
     PASS

[5] Splits: {'train', 'val', 'test'}
    train: 1363
    val: 292
    test: 293
     PASS

[6] Avg nodes/graph: 4.9  |  Avg edges/graph: 16.1
     PASS

[7] Graphs with edge_attr: 1948/1948
     PASS — edge types 0/1/2/3 present
    Graphs using edge type 3 (mentions-number): 0

[8] Edge type distribution across all graphs:
    Type 0 (sequential):  15084 (48.1%)
    Type 1 (semantic):   8822 (28.1%)
    Type 2 (value-reuse):   7460 (23.8%)
     PASS

   ALL CHECKS PASSED — Stage 4 complete!
    Ready for Stage 5 (train GNN)


## Cell 8 — Manual inspection of one graph

We print the full structure of one correct graph and one wrong graph so we can visually confirm everything looks right.

In [ ]:
graphs_check = torch.load(OUTPUT_FILE, weights_only=False)
meta_check   = pd.read_csv(META_FILE)

EDGE_TYPE_NAMES = {0:'sequential', 1:'semantic', 2:'value-reuse', 3:'mentions-number'}

def print_graph(g, meta_df):
    label_text = 'CORRECT ' if g.y.item() == 1 else 'WRONG '
    print(f'trace_id={g.trace_id}  label={g.y.item()} ({label_text})  split={g.split}')
    print(f'Nodes: {g.x.shape[0]}  |  Edges: {g.edge_index.shape[1]}  |  Features: {g.x.shape[1]}')
    print()

    steps = meta_df[meta_df['trace_id'] == g.trace_id].sort_values('step_num')
    print('Steps (nodes):')
    for _, row in steps.iterrows():
        print(f'  Node {int(row["step_num"])-1}: {row["step_text"][:75]}')
    print()

    print('Edges (src -> dst  [type]):')
    srcs  = g.edge_index[0].tolist()
    dsts  = g.edge_index[1].tolist()
    attrs = g.edge_attr.squeeze().tolist() if g.edge_attr.dim() > 1 else g.edge_attr.tolist()
    shown = set()
    for s, d, a in zip(srcs, dsts, attrs):
        pair = (min(s,d), max(s,d))
        if pair not in shown:
            etype = EDGE_TYPE_NAMES.get(int(a), str(int(a)))
            print(f'  {s} -> {d}  [{etype}]')
            shown.add(pair)

    from collections import Counter
    type_counts = Counter([EDGE_TYPE_NAMES.get(int(a), str(int(a))) for a in attrs])
    print(f'\nEdge type summary: {dict(type_counts)}')
    print()

correct_graphs = [g for g in graphs_check if g.y.item() == 1]
wrong_graphs   = [g for g in graphs_check if g.y.item() == 0]

print('=' * 65)
print('EXAMPLE 1 — Correct trace graph (label=1)')
print('=' * 65)
print_graph(correct_graphs[0], meta_check)

print('=' * 65)
print('EXAMPLE 2 — Wrong trace graph (label=0)')
print('=' * 65)
print_graph(wrong_graphs[0], meta_check)


EXAMPLE 1 — Correct trace graph (label=1)
trace_id=0  label=1 (CORRECT )  split=train
Nodes: 2  |  Edges: 2  |  Features: 768

Steps (nodes):
  Node 0: First, let's find out how many blankets Nathan added to his bed. Since he a
  Node 1: Each blanket warms up Nathan by 3 degrees. To find the total temperature in

Edges (src -> dst  [type]):
  0 -> 1  [sequential]

Edge type summary: {'sequential': 2}

EXAMPLE 2 — Wrong trace graph (label=0)
trace_id=2  label=0 (WRONG )  split=train
Nodes: 7  |  Edges: 42  |  Features: 768

Steps (nodes):
  Node 0: Let's define the total prize money as 'x'. Rica got 3/8 of the prize money,
  Node 1: Rica spent 1/5 of her prize money, which is (1/5)*(3/8)*x. To find out how 
  Node 2: Rica is left with $300 after spending 1/5 of her prize money. We can set up
  Node 3: We can simplify the equation by first finding a common denominator for (3/8
  Node 4: Combine like terms to get (15-12)/40*x = 300, which simplifies to (3/40)*x 
  Node 5: Multiply both si

##  Stage 4 Complete!

**Output file:** `data/graphs.pt`

| Property | Value |
|---|---|
| Total graphs | 292 (one per trace) |
| Node features | 768 numbers (step embedding) |
| Edge types | 3 (sequential, semantic, value-reuse) |
| Graph label | 1 = correct trace, 0 = wrong trace |
| Splits | train / val / test |

---

**What just happened:**  
Each trace's steps → nodes with 768-dim features  
3 types of edges connect the nodes  
Each graph labelled 1 or 0  
292 graphs saved to `graphs.pt`

**Next → Stage 5:**  
Train a GCN and a GAT model on these graphs to classify correct vs wrong reasoning.  
The GNN learns: what does a wrong reasoning graph look like?